In [26]:
import re
import shutil
import subprocess
import unicodedata
import urllib.request
from urllib.parse import urljoin
from pathlib import Path
import geopandas as gpd
import pandas as pd
from nume_zona import nume_zona
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "transform_data" else Path.cwd()

In [27]:
def normalize_uat_name(s: str) -> str:
    """Elimina diacriticele, cratimele si prefixul rangului administrativ (MUNICIPIUL/ORASUL/COMUNA)."""
    s = "".join(c for c in unicodedata.normalize("NFD", str(s)) if unicodedata.category(c) != "Mn")
    s = s.upper().replace("-", " ").strip()
    s = re.sub(r"^(MUNICIPIUL|ORASUL|ORAS|COMUNA)\s+", "", s)
    return re.sub(r"\s+", " ", s)

In [28]:
# venituri ( doar pentru anul >= 2025 )
DPFBL_PAGE_URL = "http://www.dpfbl.mdrap.ro/sit_ven_si_chelt_uat.html"
URL_VENITURI_2025 = "http://www.dpfbl.mdrap.ro/venituri_si_cheltuieli_uat/Anexa24_2025.xlsx"


def gaseste_url_venituri(an: int = 2025) -> str:
    """Returneaza URL-ul Excel pentru anul `an` (doar >= 2025)."""
    an = int(an)
    if an < 2025:
        raise ValueError(f"Anul trebuie sa fie >= 2025 (ai introdus {an}).")
    if an == 2025:
        return URL_VENITURI_2025

    req = urllib.request.Request(DPFBL_PAGE_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=20) as resp:
        html = resp.read().decode("utf-8", errors="ignore")
    hrefs = re.findall(r'href=["\']([^"\']+\.xlsx?)["\']', html, flags=re.IGNORECASE)
    for href in hrefs:
        if re.search(rf"(?<!\d){an}(?!\d)", href):
            return urljoin(DPFBL_PAGE_URL, href)

    raise ValueError(f"Fisierul Excel pentru anul {an} nu a fost publicat inca pe {DPFBL_PAGE_URL}.")


def descarca_venituri(an: int = 2025, fortat: bool = False) -> pd.DataFrame:
    """Descarca (daca nu exista deja pentru anul `an` >= 2025) Excel-ul de venituri si extrage judetul activ."""
    an = int(an)
    if an < 2025:
        raise ValueError(f"Anul trebuie sa fie >= 2025 (ai introdus {an}).")

    venituri_path = PROJECT_ROOT / "raw_data" / "venituri_per_uat.xlsx"
    venituri_path.parent.mkdir(parents=True, exist_ok=True)

    df_raw = None
    if not fortat and venituri_path.exists():
        try:
            candidat = pd.read_excel(venituri_path, sheet_name="Sheet2", usecols=[0, 1])
            if candidat.iloc[:5, 0].astype(str).str.contains(str(an)).any():
                df_raw = candidat
        except Exception:
            df_raw = None

    if df_raw is None:
        url = gaseste_url_venituri(an)
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=60) as resp:
            venituri_path.write_bytes(resp.read())
        df_raw = pd.read_excel(venituri_path, sheet_name="Sheet2", usecols=[0, 1])

    judet_norm = normalize_uat_name(nume_zona())
    venituri = df_raw.iloc[5:, :2].copy()
    venituri.columns = ["Cod fiscal - UAT", "Venituri totale"]
    venituri = venituri.dropna(subset=["Cod fiscal - UAT"]).reset_index(drop=True)

    judet_mask = venituri["Cod fiscal - UAT"].apply(normalize_uat_name).str.contains(
        rf"^JUDETUL:\s*{re.escape(judet_norm)}$"
    )
    judet_index = venituri.index[judet_mask][0]
    rest_venituri = venituri.iloc[judet_index + 1:]
    next_judet = rest_venituri.index[
        rest_venituri["Cod fiscal - UAT"].astype(str).str.contains("Judetul:", case=False)
    ]
    end_index = next_judet[0] if len(next_judet) > 0 else len(venituri)

    venituri_judet = venituri.iloc[judet_index + 8:end_index].reset_index(drop=True)
    venituri_judet = venituri_judet.loc[
        venituri_judet["Cod fiscal - UAT"].astype(str).str.contains("-")
    ].copy()
    venituri_judet["UAT"] = (
        venituri_judet["Cod fiscal - UAT"].str.split("-", n=1).str[1].apply(normalize_uat_name)
    )
    venituri_judet["Venituri totale"] = venituri_judet["Venituri totale"].astype(float).round(2)
    return venituri_judet.reset_index(drop=True)


venituri_judet = descarca_venituri(an=2025)

In [29]:
# populatia
def _gaseste_rscript() -> str:
    rscript = shutil.which("Rscript")
    if rscript:
        return rscript
    r_dir = Path(r"C:\Program Files\R")
    if r_dir.exists():
        candidati = sorted(r_dir.glob("R-*/bin/Rscript.exe"), reverse=True)
        if candidati:
            return str(candidati[0])
    raise FileNotFoundError("Nu a fost gasit Rscript.exe.")


def tempo_pop(an: int = 2025, fortat: bool = False) -> pd.DataFrame:
    """Ruleaza transform_data/tempo_pop.r doar daca lipsesc datele pentru anul `an` si extrage judetul activ."""
    an = int(an)
    pop_path = PROJECT_ROOT / "raw_data" / "populatie_per_uat.xlsx"

    pop = None
    if not fortat and pop_path.exists():
        try:
            candidat = pd.read_excel(pop_path)
            if "An" in candidat.columns and int(candidat["An"].iloc[0]) == an:
                pop = candidat
        except Exception:
            pop = None

    if pop is None:
        script_r = PROJECT_ROOT / "transform_data" / "tempo_pop.r"
        subprocess.run([_gaseste_rscript(), str(script_r), str(an)], check=True)
        pop = pd.read_excel(pop_path)

    judet_norm = normalize_uat_name(nume_zona())

    pop = pop[pop["Judet"].apply(normalize_uat_name) == judet_norm].copy()
    pop = pop.rename(columns={"SIRUTA_UAT": "SIRUTA UAT"}).reset_index(drop=True)
    pop["Populatie"] = pop["Populatie"].astype(float)
    pop["SIRUTA"] = pop["SIRUTA"].astype(str)
    pop["UAT"] = pop["UAT"].apply(normalize_uat_name)
    pop = pop[["SIRUTA UAT", "Populatie", "SIRUTA", "UAT"]]

    return pop


pop = tempo_pop(an=2025)

In [30]:
# venituri + populatie
venit_pop = venituri_judet.merge(pop, on="UAT", how="right")
venit_pop = venit_pop[["UAT", "SIRUTA", "Venituri totale", "Populatie"]]
venit_pop["Venit per capita"] = (venit_pop["Venituri totale"] / venit_pop["Populatie"]).round(2)
venit_pop["Economie Punctaj"] = venit_pop["Venit per capita"].rank(pct=True) * 100

In [31]:
# uat
uat_path = PROJECT_ROOT / "data" / "uat.gpkg"
uat = gpd.read_file(uat_path)
uat = uat.loc[:, ["natCode", "geometry"]].rename(columns={"natCode": "SIRUTA"})
uat["SIRUTA"] = uat["SIRUTA"].astype(str)

In [32]:
# uat + venit_pop -> salvare in data/uat_economic.gpkg
uat_venit_pop = uat.merge(venit_pop, on="SIRUTA", how="right")
out_gpkg = PROJECT_ROOT / "data" / "uat_economic.gpkg"
out_gpkg.parent.mkdir(parents=True, exist_ok=True)
uat_venit_pop.to_file(out_gpkg, driver="GPKG")
uat_venit_pop

,SIRUTA,geometry,UAT,Venituri totale,Populatie,Venit per capita,Economie Punctaj
0,26564,"POLYGON ((266913.289 628653.259, 266914.642 62...",ORADEA,1.541970e+09,213622.0,7218.22,81.188119
1,26804,"POLYGON ((300331.015 577850.794, 300336.288 57...",BEIUS,6.854700e+07,10447.0,6561.40,74.257426
2,26877,"POLYGON ((300431.857 662724.615, 300453.734 66...",MARGHITA,7.572727e+07,16448.0,4604.04,42.574257
3,26975,"POLYGON ((251467.753 595213.833, 251461.27 595...",SALONTA,1.270301e+08,17931.0,7084.39,80.198020
4,26699,"POLYGON ((306197.443 631386.796, 306184.802 63...",ALESD,1.278562e+08,10927.0,11700.94,99.009901
...,...,...,...,...,...,...,...
96,31841,"POLYGON ((256924.649 595195.986, 256927.326 59...",TULCA,1.090648e+07,2721.0,4008.26,28.712871
97,31921,"POLYGON ((290012.274 583520.855, 290000.589 58...",UILEACU DE BEIUS,1.621989e+07,1829.0,8868.17,91.089109
98,31976,"POLYGON ((313830.62 616473.943, 313835.794 616...",VADU CRISULUI,1.245995e+07,3881.0,3210.50,11.881188
99,32090,"POLYGON ((290531.105 614892.07, 290731.349 614...",VARCIOROG,9.107323e+06,1838.0,4955.02,55.445545
